# Example notebook for transfer learning

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torchvision
from torchvision.models.resnet import ResNet34_Weights, ResNet50_Weights

# Add parent directory to path so we can import src modules
parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src.training import LHmode_classifier as lh

# Setup
path = Path('../')
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    
# Load data
shots_testing, shots_validation, shots_training = lh.load_shot_data(ris_option='RIS1', 
                                                                    test_df_contains_val_df=True, 
                                                                    test_run=True, #!!! Significantly reduces amount of data involved
                                                                    data_frac=1.0, 
                                                                    random_seed=42)
    
# Create dataloaders
dataloaders, dataset_sizes, test_dataloader = lh.create_dataloaders(
    path, shots_training, shots_testing, shots_validation,
    'RIS1', num_classes=3, exponential_elm_decay=True,
    batch_size=16, num_workers=4, augmentation=False, grayscale=False)
    
# Setup model
pretrained_model = torchvision.models.resnet18()
model = lh.setup_model(pretrained_model, num_classes=3, device=device, grayscale=False)
    
# Train only FC layer
model = lh.train_phase(
    model, dataloaders, dataset_sizes, 
    timestamp='manual_example', path=path, phase='last_fc',
    num_epochs=2, learning_rate_min=0.001, learning_rate_max=0.01,
    weight_decay=1e-4, freeze_backbone=True)

TypeError: load_shot_data() got multiple values for argument 'ris_option'

In [ ]:
# === METHOD 2: Detailed step-by-step ===
print("\nMethod 2: Detailed training (for full control)")
print("Each step is explicit, allowing customization")

# Setup
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    
# Load data - LHmode_classifier automatically detects project root
shots_testing, shots_validation, shots_training = lh.load_shot_data(
    ris_option='RIS1', 
    test_df_contains_val_df=True, 
    test_run=True, #!!! Significantly reduces amount of data involved
    data_frac=1.0, 
    random_seed=42)
    
# Create dataloaders
dataloaders, dataset_sizes, test_dataloader = lh.create_dataloaders(
    shots_training, shots_testing, shots_validation,
    'RIS1', num_classes=3, exponential_elm_decay=True,
    batch_size=16, num_workers=4, augmentation=False, grayscale=False)
    
# Setup model
pretrained_model = torchvision.models.resnet18()
model = lh.setup_model(pretrained_model, num_classes=3, device=device, grayscale=False)
    
# Train only FC layer
model_detailed = lh.train_phase(
    model, dataloaders, dataset_sizes, 
    timestamp='detailed_demo', phase='last_fc',
    num_epochs=2, learning_rate_min=0.001, learning_rate_max=0.01,
    weight_decay=1e-4, freeze_backbone=True)

print("\n" + "="*50)
print("Method 2 completed! Check runs/ directory for results.")
print("Both methods automatically save to /compass/Shared/Users/bogdanov/ml_tokamak/runs/")
print("="*50)